All matrices used in my code, with their shapes and entries.

## 1. mesh_maker.py

Mesh:
- `L`: length of the pool
- `H`: depth of the pool
- `n_spines`: number of spines
- `layers`: number of layers of triangles; two triangles = 1 layer
- `nodes_per_col`: the number of rows
- `num_elements`: total number of triangles
- `num_global_nodes`: total number of nodes
- `num_p_nodes`: number of pressure nodes

**Matrices**
- `loc_to_glob`: (num_elements, 6) - an entry $(e, ii)$ returns the global node of the local node $ii$ of element $e$.
\begin{bmatrix}
0 & 10 & 12 & 6 & 5 & 11 \\
0 & 12 & 2 & 1 & 6 & 7 \\
2 & 12 & 14 & 8 & 7 & 13 \\
2 & 14 & 4 & 3 & 8 & 9 \\
\vdots & \vdots & \vdots & \vdots & \vdots & \vdots \\
82 & 92 & 94 & 88 & 87 & 93 \\
82 & 94 & 84 & 83 & 88 & 89
\end{bmatrix}_{36 \times 6}

- `loc_to_glob_p`: (num_elements, 3)
\begin{bmatrix}
0 & 10 & 12 \\
0 & 12 & 2 \\
2 & 12 & 14 \\
2 & 14 & 4 \\
\vdots & \vdots & \vdots \\
82 & 92 & 94 \\
82 & 94 & 84
\end{bmatrix}_{36 \times 3}

- `surface_nodes`: (19, 1) - the nodes on the free surface

\begin{bmatrix}
4 \\
9 \\
\vdots \\
89 \\
94
\end{bmatrix}_{19 \times 1}

- `loc_to_glob_1`: (9, 3) - a matrix, where the $i$-th row represents the three nodes of the line element (left, mid, right).
\begin{bmatrix}
4 & 9 & 14 \\
14 & 19 & 24 \\
\vdots & \vdots & \vdots \\
74 & 79 & 84 \\
84 & 89 & 94
\end{bmatrix}_{9 \times 3}

- `coords`: (num_global_nodes, 2) - returns the physical positions of that node
\begin{bmatrix}
x_0 & z_0 \\
x_1 & z_1 \\
x_2 & z_2 \\
\vdots & \vdots \\
x_{93} & z_{93} \\
x_{94} & z_{94}
\end{bmatrix}_{94 \times 2}

**Unknowns**
- `u_unknown` = `w_unknown`: (num_global_nodes - nodes on the wall, 1) - vectors representing the nodes with unknown u and w values (all global nodes except for the ones on the wall)
\begin{bmatrix}
6 \\
7 \\
\vdots \\
88 \\
89
\end{bmatrix}_{68 \times 1}
- `p_unknown`: (num_p_nodes, 1) - vector representing all nodes with unknown p
\begin{bmatrix}
0 \\
1 \\
\vdots \\
28 \\
29
\end{bmatrix}_{30 \times 1}
- `surface_unknown`: (17, 1) - vector representing all surface nodes except for the ones pinned to the walls
\begin{bmatrix}
9 \\
14 \\
\vdots \\
84 \\
89
\end{bmatrix}_{17 \times 1}

## 2. quadrature.py
- `n_Gaussian_Q = 16` <-- number of Gaussian quadrature points on the master element (bulk integrals)
- `n_lGaussian_Q = 11`  <-- number of Gaussian quadrature points on the master interval [-1, 1] (line integrals along the free surface)

**Matrices:**
- `W_G` : (16, 1) - the weights
\begin{bmatrix}
0.28863122 \\
0.20643474 \\
\vdots \\
0.05446063 \\
0.05446063
\end{bmatrix}_{16 \times 1}

- `W_lG`: (11, 1) - the weights for the line Gaussian quadrature terms
\begin{bmatrix}
0.05566857 \\
0.12558037 \\
\vdots \\
0.12558037 \\
0.05566857
\end{bmatrix}_{11 \times 1}




- `xi_eta_G`: (16, 2) - row $p$ holds the $(\xi_p, \eta_p)$ coordinates of the $p$-th Gaussian quadrature point on the master element.

\begin{bmatrix}
-0.33333333 & -0.33333333 \\
-0.65886138 & 0.31772277 \\
\vdots & \vdots \\
-0.47377434 & -0.98321045 \\
-0.98321045 & -0.47377434
\end{bmatrix}_{16 \times 2}
The first column is also `xi_G`, and the second column is `eta_G`.

- `diff_xi_phi`: (6, n_Gaussian_Q) - an entry $(ii, pp)$ $\partial_\xi \phi_{ii}$ correspondingto the Gaussian quadrature point $pp$.

\begin{bmatrix}
-0.167 & 0.159 & \ldots & -0.957 & -0.957 \\
-0.167 & 0.159 & \ldots & 0.026 & -0.483 \\
0 & 0 & \ldots & 0 & 0 \\
-0.667 & -1.31 & \ldots & 0.017 & -0.526 \\
-0.000 & 0.000 & \ldots & -0.931 & 1.440 \\
0.667 & 1.318 & \ldots & 0.0168 & 0.526 
\end{bmatrix}_{6 \times 16}

- `diff_eta_phi`:  (6, n_Gaussian_Q) - an entry $(ii, pp)$ $\partial_\eta \phi_{ii}$ correspondingto the Gaussian quadrature point $pp$. The shape is similar to `diff_xi_phi`.

- `T`: (n_Gaussian_Q, 6, 6) - calculated using $T_{ii, jj} = \partial_\xi \phi_{ii} \times \partial_\eta \phi_{jj} - \partial_\eta \phi_{ii} \times \partial_\xi \phi_{jj}$ for every quadrature point $pp$. An entry $(pp, ii, jj)$ gives the result of the formula above for $\phi$ functions indexed $ii$ and $jj$ at the quadrature point $pp$.

- `psi_G`: (3, n_Gaussian_Q) - master element pressure basis functions $\psi$. Columns sum up to 1.

\begin{bmatrix}
0.333 & 0.171 & \ldots & 0.729 & 0.729 \\
0.333 & 0.171 & \ldots & 0.263 & 0.008 \\
0.333 & 0.659 & \ldots & 0.008 & 0.263 
\end{bmatrix}_{3 \times 16}

- `phi_G`: (6, n_Gaussian_Q) - master element velocity basis functions $\phi$.

\begin{bmatrix}
-0.111 & -0.112 & \ldots & 0.333 & 0.333 \\
-0.111 & -0.112 & \ldots & -0.125 & -0.008 \\
-0.111 & 0.209 & \ldots & -0.008 & -0.125 \\
0.444 & 0.450 & \ldots & 0.024 & 0.767 \\
0.444 & 0.116 & \ldots & 0.767 & 0.024 \\
0.444 & 0.450 & \ldots & 0.009 & 0.009
\end{bmatrix}_{6 \times 16}

- `phi1_lG`: (3, n_lGaussian_Q) - line-element functions $\phi$ at the quadrature points for boundary 1.

\begin{bmatrix}
0.968 & 0.837 & \ldots & -0.050 & -0.011 \\
0.043 & 0.213 & \ldots & 0.213 & 0.043 \\
-0.011 & -0.050 & \ldots & 0.837 & 0.968 
\end{bmatrix}_{3 \times 11}

- `diff_xi_phi1_lG`: (3, n_lGaussian_Q) - dervatives of those function w.r.t $\xi$.
\begin{bmatrix}
-1.478 & -1.387 & \ldots & 0.387 & 0.478 \\
1.956 & 1.774 & \ldots & -1.774 & -1.956 \\
-0.478 & -0.387 & \ldots & 1.387 & 1.478 
\end{bmatrix}_{3 \times 11}

## 3. residual.py

All the residuals.
- `Mx`, `Mz`, `K`: (num_global_nodes, 1) - vectors that store the values of the x-momentum, z-momentum, and KBC residuals at each node.
\begin{bmatrix}
M^x_1 \\ M^x_2 \\ M^x_3 \\ \vdots \\ M^x_{94} \\ M^x_{95}
\end{bmatrix}_{95 \times 1} 

- `C`: (num_p_nodes, 1) - a vector that stores the value of the continuity residual at each node.
\begin{bmatrix}
C_1 \\ C_2 \\ C_3 \\ \vdots \\ C_{29} \\ C_{30}
\end{bmatrix}_{30 \times 1}


**Bulk terms:**
- `nodes`: (6, 1) - returns 6 global nodes of the element $ee$
- `x_e`, `z_e`: (6, 1) - x and z coordinaes of the element nodes; first and second columns of the coordinates of `nodes`, respectively
\begin{bmatrix}
0 \\ 0.111 \\ 0.111 \\ 0.056 \\ 0.056 \\ 0.111
\end{bmatrix}_{6 \times 1}
- `x_dot_e`, `z_dot_e`: (6, 1) - $\dot{\mathcal{X}} = \dfrac{x_e^{n+1} - x_e^n}{\Delta t}, \dot{\mathcal{Z}} = \dfrac{z_e^{n+1} - z_e^n}{\Delta t}$
- `u_e`, `w_e`: (6, 1) - velocities at the element nodes.
- `u_dot_e`, `w_dot_e`: (6, 1) - $\dot{u} = \dfrac{u_e^{n+1} - u_e^n}{\Delta t}, \dot{w} = \dfrac{w_e^{n+1} - w_e^n}{\Delta t}$
- `det_Je`: (n_Gaussian_Q, 1) - $\det J_e = \sum_{ii=1}^6 \sum_{jj=1}^6 \mathcal{X}_{l(e, ii)}(t) T_{ii, jj} \mathcal{Z}_{l(e, jj)}(t)$.
\begin{bmatrix}
0.007 \\ 0.007 \\ 0.007 \\ \vdots \\ 0.007 \\ 0.007
\end{bmatrix}_{16 \times 1}
Calculated with `det_Je[pp] = x_e @ T[pp] @ z_e` for each Gaussian quadrature point. Recall the dimensions: $x_e: (6, 1); T: (16, 6, 6), z_e: (6, 1)$. `@` performs matrix multiplication $x^T_e T z_e$ as follows
$$
\begin{bmatrix}
x_e^1 & x_e^2 & x_e^3 & x_e^4 & x_e^5 & x_e^6
\end{bmatrix}
\begin{bmatrix}
T_{1, 1} & \cdots & T_{1, 6} \\
\vdots & \ddots & \vdots \\
T_{6, 1} & \cdots & T_{6, 6}
\end{bmatrix}
\begin{bmatrix}
z_e^1 \\ \vdots \\ z_e^6
\end{bmatrix}
$$
and returns a scalar that represents the determinant for the quadrature point $pp$.

**Line integral terms:**

Similar to the bulk terms, but instead of taking the nodes from `loc_to_glob`, we take them from `loc_to_glob_1` matrix. The 

## 4. jacobian.py

**$x$ and $z$-momentum derivatives:**
- `diff_u_Mx`: (num_global_nodes, num_global_nodes) - An entry $(i, j)$ represents $\dfrac{\partial M_i^x}{\partial u_j}$, which tells us how much the $x$-momentum residual at node $i$ changes when we slightly nudge the horizontal velocity $u$ at global node $j$.
- The same applies for `diff_w_Mx`.
- `diff_p_Mx`: (num_global_nodes, num_p_nodes) - An entry $(i, j)$ represents $\dfrac{\partial M_i^x}{\partial p_j}$, which tells us how much the $x$-momentum residual at node $i$ changes when we nudge the pressure $p$ at global node $j$. That's why the number of columns is the number of pressure nodes.
- `diff_h_Mx`: (num_global_nodes, n_spines - 2) - An entry $(i, j)$ represents $\dfrac{\partial M_i^x}{\partial h_j}$, which tells us how much the $x$-momentum residual at node $i$ changes when we change the spine height $h$ at global node $j$. There are n_spines spines in total, but the fluid is pinned at both rims, so the spines on the walls don't change. Hence, n_spines - 2 columns.

$z$-momentum and KBC matrices have the exact same shapes, except KBC doesn't depend on pressure.

- **Continuity derivatives:**
- `diff_u_C`: (num_p_nodes, num_global_nodes) - An entry $(i, j)$ represents $\dfrac{\partial C_i}{\partial u_j}$, which tells us how much the continuity residual at node $i$ changes when we slightly nudge the horizontal velocity $u$ at global node $j$. Since the continuity equation is weighed only with pressure basis functions $\psi_i$, we get one row per pressure node. Hence num_p_nodes rows.
- The same applies for `diff_w_C`.
- `diff_h_C` columns have the same idea as $\partial_h M^x$ columns, but the number of rows is num_p_nodes.


### Other derivatives and vectors
- `diff_h_z`: (num_global_nodes, 1) - $\partial_h \mathcal{Z} = \dfrac{\text{row}_i}{\#\text{ of rows} - 1}$. `diff_h_z_e` holds the derivatives $\partial_h \mathcal{Z}_i$ for the specified global nodes $i$.
- `spine`: (num_global_nodes, 1) - entry $i$ returns the spine index of the global node $i$. `spines_e` returns the indices of spines for the nodes of the selected element.
TODO: document the following, in particular the spine logic
- `diff_det_Je`: (n_Gaussian_Q, 1) - $\partial_{h} \det J_e$
- `diff_h_Je_1`: (n_lGaussian_Q, 1) - $\partial_{h} J_{e_1}^1$
- `diff_hq_diff_xi_z_l`: (n_lGaussian_Q, 3) - $\partial_{h}\big(\partial_\xi z_{e_1}^1\big)$